In [1]:
from pathlib import Path

def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit("ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = data_dir()

# Solution — Module 2: Feature Engineering

**Dataset:** *Combined Cycle Power Plant* — Tüfekci, P. & Kaya, H. (2014). UCI ML Repository. https://doi.org/10.24432/C5002N (CC BY 4.0)

In [2]:
import polars as pl
import numpy as np

ccpp = pl.read_csv(DATA / "ccpp_power_plant.csv")
X = ccpp.select("AT", "V", "AP", "RH")
y = ccpp["PE"]

## ข้อ 1 — split ก่อน

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"train {len(X_train)} | test {len(X_test)}")

train 7654 | test 1914


## ข้อ 2 — scaler + ridge

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])
pipe.fit(X_train, y_train)
r2_train = pipe.score(X_train, y_train)
r2_test = pipe.score(X_test, y_test)
print(f"train R² = {r2_train:.4f} | test R² = {r2_test:.4f}")

train R² = 0.9283 | test R² = 0.9301


train ≈ test → โมเดลเชิงเส้นธรรมดาไม่ overfit อยู่แล้ว จุดที่ feature engineering ช่วยได้คือดัน test R² ขึ้น ไม่ใช่ลดช่องว่าง

## ข้อ 3 — เพิ่ม polynomial

In [5]:
from sklearn.preprocessing import PolynomialFeatures

pipe_poly = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge(alpha=1.0)),
])
pipe_poly.fit(X_train, y_train)
r2_poly = pipe_poly.score(X_test, y_test)
print(f"poly(2)+ridge test R² = {r2_poly:.4f}")

poly(2)+ridge test R² = 0.9383


**ทำไม scaler ต้องมาก่อน poly:** ถ้ายกกำลังสองก่อน `AP` (ราว 1000) จะกลายเป็น ~1,000,000 — สเกลของ feature ยกกำลังต่างกันมาก ทำให้ regularization ลงโทษไม่สม่ำเสมอ การ scale ก่อนทำให้ทุกเทอมอยู่สเกลเดียวกันก่อนแล้วค่อยยกกำลัง

## ข้อ 4 — GridSearchCV

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {"ridge__alpha": [0.01, 0.1, 1, 10, 100]}
search = GridSearchCV(pipe_poly, param_grid, cv=5)
search.fit(X_train, y_train)
best_alpha = search.best_params_["ridge__alpha"]
r2_best = search.best_estimator_.score(X_test, y_test)
print(f"best alpha = {best_alpha} | test R² = {r2_best:.4f}")

best alpha = 1 | test R² = 0.9383


สังเกตว่า GridSearchCV รัน CV **บน train เท่านั้น** — test ยังคงสะอาด เพราะเราไม่เคยปรับ alpha โดยดูผล test

## ข้อ 5 — domain features

In [7]:
def dew_point(at, rh):
    a, b = 17.62, 243.12
    gamma = np.log(rh / 100) + a * at / (b + at)
    return b * gamma / (a - gamma)

X2_train = X_train.with_columns(
    pl.Series("AT_x_RH", X_train["AT"].to_numpy() * X_train["RH"].to_numpy()),
    pl.Series("dew_point", dew_point(X_train["AT"].to_numpy(), X_train["RH"].to_numpy())),
)
X2_test = X_test.with_columns(
    pl.Series("AT_x_RH", X_test["AT"].to_numpy() * X_test["RH"].to_numpy()),
    pl.Series("dew_point", dew_point(X_test["AT"].to_numpy(), X_test["RH"].to_numpy())),
)

pipe_domain = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge(alpha=best_alpha)),
])
pipe_domain.fit(X2_train, y_train)
r2_domain = pipe_domain.score(X2_test, y_test)
print(f"poly + domain features test R² = {r2_domain:.4f}")

poly + domain features test R² = 0.9393


โดยทั่วไปผลจะขยับขึ้นเล็กน้อยหรือเท่าเดิม — `dew_point` สัมพันธ์กับ AT/RH แน่นจึงให้ข้อมูลใหม่น้อย บทเรียนจริง: **domain feature ไม่ได้ magic ทุกครั้ง** — วัดผลด้วย test R² เสมอ

## ข้อ 6 — คำตอบ

1. **fit scaler กับทั้งชุดก่อน split** — test set จะถูก normalize ด้วยสถิติที่รวมตัวมันเองไปแล้ว ตัวเลข test R² มักสวยขึ้นเล็กน้อยโดยไม่สะท้อนความสามารถจริงกับข้อมูลใหม่ ในงานเมโทรโลยีเทียบได้กับการสอบเทียบเครื่องมือโดยใช้ข้อมูลจากตัวอย่างที่จะใช้ตรวจ — ผลรับรองไม่มีความหมาย
2. **ไม่ต้อง scale** — ต้นไม้ตัดสินใจด้วย threshold ต่อคอลัมน์ (ทำนายจาก `AT > 20.5` เป็นต้น) ซึ่งไม่ได้รับผลจากสเกล ต่างจาก ridge/KNN/SVM/PCA ที่วัดระยะทางหรือถูกลงโทษตามขนาดสัมประสิทธิ์